In [1]:
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from src.utils import generate_mask_tensor
from src.embedding import embed
from src.gp_ccm import GP_ccm_sig, run_sigGPCCM_experiment
from src.sp_ccm import run_SP_CCM, SP_CCM_iaaft, run_ccm_experiment
from src.iaaft import surrogates

from scipy.stats import ranksums
torch.set_printoptions(sci_mode = False)

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



# Data generation

In [3]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
c = torch.tensor([0.2])
a = torch.tensor([0.3])
b = torch.tensor([0.4])

# Autoregressive function
for t in range(N_length - 1):
    
    c_next = c[t] * (3.9 - (3.9 * c[t]))
    a_next = a[t] * (3.8 - (3.8 * a[t]) - (0.3 * c[t])) # previously 0.2
    b_next = b[t] * (3.6 - (3.6 * b[t]) - (0.2 * c[t]))

    c = torch.concat((c, c_next.unsqueeze(0)))
    a = torch.concat((a, a_next.unsqueeze(0)))
    b = torch.concat((b, b_next.unsqueeze(0)))
            
# Normalising step
c_norm = c.sub(c.mean(dim = -1).unsqueeze(-1)).div(c.std(dim = -1).unsqueeze(-1))
a_norm = a.sub(a.mean(dim = -1).unsqueeze(-1)).div(a.std(dim = -1).unsqueeze(-1))
b_norm = b.sub(b.mean(dim = -1).unsqueeze(-1)).div(b.std(dim = -1).unsqueeze(-1))

# In this setting we (mostly) are better!

In [20]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
c = torch.tensor([0.2])
a = torch.tensor([0.3])
b = torch.tensor([0.4])

# Autoregressive function
for t in range(N_length - 1):
    
    c_next = c[t] * (3.9 - (3.9 * c[t]))
    a_next = a[t] * (3.8 - (3.8 * a[t]) - (0.4 * c[t])) # previously 0.2
    b_next = b[t] * (3.6 - (3.6 * b[t]) - (0.3 * c[t]))

    c = torch.concat((c, c_next.unsqueeze(0)))
    a = torch.concat((a, a_next.unsqueeze(0)))
    b = torch.concat((b, b_next.unsqueeze(0)))
            
# Normalising step
c_norm = c.sub(c.mean(dim = -1).unsqueeze(-1)).div(c.std(dim = -1).unsqueeze(-1))
a_norm = a.sub(a.mean(dim = -1).unsqueeze(-1)).div(a.std(dim = -1).unsqueeze(-1))
b_norm = b.sub(b.mean(dim = -1).unsqueeze(-1)).div(b.std(dim = -1).unsqueeze(-1))

# In this setting we  are better!

In [26]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
c = torch.tensor([0.2])
a = torch.tensor([0.3])
b = torch.tensor([0.4])

# Autoregressive function
for t in range(N_length - 1):
    
    c_next = c[t] * (3.9 - (3.9 * c[t]))
    a_next = a[t] * (3.8 - (3.8 * a[t]) - (0.3 * c[t])) # previously 0.2
    b_next = b[t] * (3.6 - (3.6 * b[t]) - (0.3 * c[t]))

    c = torch.concat((c, c_next.unsqueeze(0)))
    a = torch.concat((a, a_next.unsqueeze(0)))
    b = torch.concat((b, b_next.unsqueeze(0)))
            
# Normalising step
c_norm = c.sub(c.mean(dim = -1).unsqueeze(-1)).div(c.std(dim = -1).unsqueeze(-1))
a_norm = a.sub(a.mean(dim = -1).unsqueeze(-1)).div(a.std(dim = -1).unsqueeze(-1))
b_norm = b.sub(b.mean(dim = -1).unsqueeze(-1)).div(b.std(dim = -1).unsqueeze(-1))

# Visualise

In [28]:
fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, c_norm.shape[0])[0:41], y = c[0:41],
                    mode = 'lines',
                    name = 'C',
                    line_color = "forestgreen"))

fig.add_trace(go.Scatter(x = torch.arange(0, c_norm.shape[0])[0:41], y = a[0:41],
                    mode = 'lines',
                    name = 'A',
                    line_color = "blue"))

fig.add_trace(go.Scatter(x = torch.arange(0, c_norm.shape[0])[0:41], y = b[0:41],
                    mode = 'lines',
                    name = 'B',
                    line_color = "cornflowerblue"))

fig.update_layout(title = 'Confounding time series',
                   xaxis_title = 't',
                   yaxis_title = 'values')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(xaxis_range=[-2,41])

fig.update_layout(autosize = False, width = 1000, height = 400)

fig.show()

In [29]:
# GLOBALS
k = 3
N_TRAIN = torch.tensor([100]).to(device)

##############
### GP-CCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) # k -1 

NOISE_SCALE = torch.tensor([0.05], device = device) # for diagonal
RBF_SCALE = torch.tensor([0.3], device = device)

############
### ECCM ###
############

ccm_filter = torch.ones(size = (k, )).to(device) # same as sig filter
ccm_shift = torch.tensor(ccm_filter.shape[0] - 1).to(device)

# C -> A

In [30]:
### sig-GP_CCM ###
CA_gpccm_rho_mean, CA_gpccm_rho_sd, CA_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = c_norm.to(device),
    causal_y = a_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.721
Rho std 0.051
Rho indep. p95 0.098


In [31]:
### CCM ###
CA_ccm_rho_mean, CA_ccm_rho_sd, CA_ccm_rho_ind_p95, CA_ccm_noise =  run_ccm_experiment(
    causal_x = c_norm.to(device),
    causal_y = a_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.819
Rho std 0.021
Rho indep. p95 0.126
Added noise 0.0


# C -> B

In [32]:
### sig-GP_CCM ###
CB_gpccm_rho_mean, CB_gpccm_rho_sd, CB_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = c_norm.to(device),
    causal_y = b_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.759
Rho std 0.082
Rho indep. p95 0.113


In [33]:
### CCM ###
CB_ccm_rho_mean, CB_ccm_rho_sd, CB_ccm_rho_ind_p95, CB_ccm_noise =  run_ccm_experiment(
    causal_x = c_norm.to(device),
    causal_y = b_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.904
Rho std 0.016
Rho indep. p95 0.167
Added noise 0.0


# A -> B

In [34]:
### sig-GP_CCM ###
AB_gpccm_rho_mean, AB_gpccm_rho_sd, AB_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = a_norm.to(device),
    causal_y = b_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean -0.006
Rho std 0.064
Rho indep. p95 0.108


In [35]:
### CCM ###
AB_ccm_rho_mean, AB_ccm_rho_sd, AB_ccm_rho_ind_p95, AB_ccm_noise =  run_ccm_experiment(
    causal_x = a_norm.to(device),
    causal_y = b_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.091
Rho std 0.117
Rho indep. p95 0.125
Added noise 0.0


# A -> C

In [36]:
### sig-GP_CCM ###
AC_gpccm_rho_mean, AC_gpccm_rho_sd, AC_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = a_norm.to(device),
    causal_y = c_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean -0.017
Rho std 0.106
Rho indep. p95 0.119


In [37]:
### CCM ###
AC_ccm_rho_mean, AC_ccm_rho_sd, AC_ccm_rho_ind_p95, AC_ccm_noise =  run_ccm_experiment(
    causal_x = a_norm.to(device),
    causal_y = c_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

We get nan's and have to increase the noise level.
Rho mean 0.077
Rho std 0.062
Rho indep. p95 0.139
Added noise 0.025


# B -> A

In [38]:
### sig-GP_CCM ###
BA_gpccm_rho_mean, BA_gpccm_rho_sd, BA_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = b_norm.to(device),
    causal_y = a_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.073
Rho std 0.049
Rho indep. p95 0.098


In [39]:
### CCM ###
BA_ccm_rho_mean, BA_ccm_rho_sd, BA_ccm_rho_ind_p95, BA_ccm_noise =  run_ccm_experiment(
    causal_x = b_norm.to(device),
    causal_y = a_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.159
Rho std 0.053
Rho indep. p95 0.109
Added noise 0.0


# B -> C

In [40]:
### sig-GP_CCM ###
BC_gpccm_rho_mean, BC_gpccm_rho_sd, BC_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = b_norm.to(device),
    causal_y = c_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.098
Rho std 0.037
Rho indep. p95 0.11


In [42]:
### CCM ###
BC_ccm_rho_mean, BC_ccm_rho_sd, BC_ccm_rho_ind_p95, BC_ccm_noise =  run_ccm_experiment(
    causal_x = b_norm.to(device),
    causal_y = c_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

We get nan's and have to increase the noise level.
Rho mean 0.13
Rho std 0.029
Rho indep. p95 0.129
Added noise 0.025
